In [1]:
import os
import pandas as pd


In [2]:
import os
os.getcwd()

'C:\\Users\\naush\\semantic-search-system'

In [3]:
os.listdir()

['.ipynb_checkpoints',
 'data',
 'dataset_loading.ipynb',
 'Untitled.ipynb',
 'venv']

In [4]:
os.listdir("data")

['20newsgroups.data.html',
 '20newsgroups.html',
 '20_newsgroups',
 '20_newsgroups.tar.gz',
 'mini_newsgroups',
 'mini_newsgroups.tar.gz',
 'venv']

In [5]:
data_path = "data/20_newsgroups/20_newsgroups"

In [7]:
import os

data_path = "data/20_newsgroups/20_newsgroups"

documents = []
labels = []

for category in os.listdir(data_path):

    category_path = os.path.join(data_path, category)

    if os.path.isdir(category_path):

        for file in os.listdir(category_path):

            file_path = os.path.join(category_path, file)

            with open(file_path, "r", encoding="latin1") as f:
                documents.append(f.read())
                labels.append(category)

print("Total documents:", len(documents))
print("Total categories:", len(set(labels)))

Total documents: 19997
Total categories: 20


In [8]:
import pandas as pd

df = pd.DataFrame({
    "text": documents,
    "category": labels
})

df.head()

,text,category
0,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49...,alt.atheism
1,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism
2,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,alt.atheism
3,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism


In [9]:
import re

In [10]:
def clean_text(text):
    
    # Remove email headers (everything before the first blank line)
    text = re.split(r"\n\n", text, maxsplit=1)[-1]

    # Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [11]:
df["clean_text"] = df["text"].apply(clean_text)

In [12]:
print("BEFORE CLEANING:\n")
print(df["text"][0][:500])

print("\n\nAFTER CLEANING:\n")
print(df["clean_text"][0][:500])

BEFORE CLEANING:

Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49960 alt.atheism.moderated:713 news.answers:7054 alt.answers:126
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!bb3.andrew.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!magnus.acs.ohio-state.edu!usenet.ins.cwru.edu!agate!spool.mu.edu!uunet!pipex!ibmpcug!mantis!mathew
From: mathew <mathew@mantis.co.uk>
Newsgroups: alt.atheism,alt.atheism.moderated,news.answers,alt.answers
Subject: Alt.Atheism FAQ: Atheist Resources
Summary: Books, addresses, mu


AFTER CLEANING:

Archive name atheism resources Alt atheism archive name resources Last modified 11 December 1992 Version 1 0 Atheist Resources Addresses of Atheist Organizations USA FREEDOM FROM RELIGION FOUNDATION Darwin fish bumper stickers and assorted other atheist paraphernalia are available from the Freedom From Religion Foundation in the US Write to FFRF P O Box 750 Madison WI 53701 Telephone 608 256 8900 EVOLUTION DESIGNS Evolution Designs sell the Darwin fish It s 

In [13]:
df.shape

(19997, 3)

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
embeddings = model.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

In [17]:
import numpy as np

embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

Embedding shape: (19997, 384)


In [18]:
df["embedding"] = embeddings.tolist()

# text
# category
# clean_text
# embedding

# Sentence Transformers convert documents into dense vector
# representations that capture semantic meaning. These embeddings
# allow efficient similarity search and clustering in vector space.

In [19]:
# Step 7 — Build the Vector Database (FAISS)
import faiss
import numpy as np

In [20]:
embedding_matrix = np.array(embeddings).astype("float32")

In [21]:
embedding_matrix.shape

(19997, 384)

In [22]:
# Create FAISS Index
# IndexFlatL2 → exact similarity search using Euclidean distance
# dimension → embedding size (384)

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

In [23]:
index.add(embedding_matrix)

In [24]:
# Add Vectors to FAISS
print("Total vectors in index:", index.ntotal)

Total vectors in index: 19997


In [25]:
query = "space mission launch satellite"

query_embedding = model.encode([query]).astype("float32")

k = 5

distances, indices = index.search(query_embedding, k)

In [26]:
# 7.6 Display Results
# FAISS is used as a vector index to efficiently perform
# nearest-neighbour search over document embeddings.
# This enables semantic retrieval based on meaning rather
# than keyword matching.

for i in indices[0]:
    print("Category:", df.iloc[i]["category"])
    print(df.iloc[i]["clean_text"][:200])
    print("-----")

Category: sci.space
Date Tue 6 Apr 1993 15 40 47 GMT I need as much information about Cosmos 2238 and its rocket fragment 1993 018B as possible Both its purpose launch date location in short EVERYTHING Can you help Tony 
-----
Category: sci.space
In article Henry Spencer writes In article David M Palmer writes orbiting billboard I would just like to point out that it is much easier to place an object at orbital altitude than it is to place it 
-----
Category: sci.space
I can tell you that when AMSAT launched some birds along a Spot satellite French that during installation of some instruments on Spot 2 there heavily armed legionaires who had a take no prisoners look
-----
Category: sci.space
Bill Higgins Beam Jockey writes In article writes COMMERCIAL SPACE NEWS SPACE TECHNOLOGY INVESTOR NUMBER 22 Other commercial launch site ventures including those at Woomera Poker Flat Cape York White 
-----
Category: sci.space
ARIANESPACE FLIGHT 56 Flight V 56 was originally intended to carry the H

In [27]:
# Step 8 — Fuzzy Clustering of Documents

!pip install scikit-fuzzy

In [28]:
import skfuzzy as fuzz

In [29]:
# 8.3 Prepare Data for Clustering
# Fuzzy C-Means expects data in feature × samples format.We must transpose them.

X = embedding_matrix.T

In [30]:
print(X.shape)

(384, 19997)


In [31]:
n_clusters = 12

In [32]:
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    X,
    c=n_clusters,
    m=2,
    error=0.005,
    maxiter=1000,
    init=None
)

In [33]:
u

array([[0.08333324, 0.08333315, 0.08333337, ..., 0.08333331, 0.08333318,
        0.08333342],
       [0.08333382, 0.08333409, 0.08333368, ..., 0.08333402, 0.08333425,
        0.08333361],
       [0.0833328 , 0.0833325 , 0.08333292, ..., 0.08333254, 0.08333232,
        0.08333297],
       ...,
       [0.08333286, 0.08333265, 0.08333288, ..., 0.08333258, 0.08333252,
        0.08333295],
       [0.08333344, 0.08333348, 0.0833334 , ..., 0.08333346, 0.08333346,
        0.08333338],
       [0.08333356, 0.08333365, 0.08333352, ..., 0.08333365, 0.08333361,
        0.08333343]], shape=(12, 19997))

In [34]:
print(u.shape)

(12, 19997)


In [35]:
# 8.7 Assign Dominant Cluster

cluster_labels = np.argmax(u, axis=0)

In [36]:
cluster_labels[:10]

array([4, 4, 4, 4, 4, 4, 4, 4, 4, 4])

In [37]:
# 8.8 Add Cluster Labels to DataFrame

df["cluster"] = cluster_labels

In [38]:
df.head()

,text,category,clean_text,embedding,cluster
0,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49...,alt.atheism,Archive name atheism resources Alt atheism arc...,"[-0.0285569429397583, 0.016917048022150993, -0...",4
1,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism,Archive name atheism introduction Alt atheism ...,"[-0.027238793671131134, -0.04237806797027588, ...",4
2,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,alt.atheism,In article Charley Wingate writes Well John ha...,"[-0.09503985196352005, -0.004863628186285496, ...",4
3,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism,until kings become philosophers or philosopher...,"[-0.033383529633283615, 0.044307440519332886, ...",4
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism,In article Bob McGwier writes 1 HOWEVER I hate...,"[-0.02208806946873665, 0.06549318879842758, -0...",4


In [39]:
df["cluster"].value_counts()

cluster
4     9311
3     8548
5      688
9      657
0      395
7      250
8       97
2       26
11       8
10       8
1        7
6        2
Name: count, dtype: int64

In [40]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [41]:
# Fuzzy C-Means clustering is applied to the document embeddings
# to discover latent semantic groups within the dataset.

# Unlike hard clustering algorithms such as K-Means,
# Fuzzy C-Means assigns membership probabilities to each
# document across clusters, allowing documents to belong
# to multiple semantic themes simultaneously.


semantic_cache = {}

In [42]:
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
# 9.3 Build Cache Search Function

# Create a function to check if a similar query already exists.

def check_cache(query_embedding, threshold=0.90):

    for cached_query, cached_result in semantic_cache.items():

        similarity = cosine_similarity(
            [query_embedding],
            [cached_query]
        )[0][0]

        if similarity > threshold:
            return cached_result

    return None

In [44]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [46]:
model.encode(["space shuttle mission"])

array([[ 4.80819214e-03, -1.68409804e-03, -3.79324332e-02,
         1.45229530e-02, -7.44499862e-02, -1.62368976e-02,
         7.86334053e-02, -2.18448155e-02,  5.48836589e-03,
        -1.33092189e-02, -2.20028367e-02,  2.02864539e-02,
         1.31824063e-02, -1.10626984e-02, -2.69972831e-02,
        -1.24883000e-02,  6.51932359e-02, -6.81476714e-03,
        -4.14763875e-02, -6.11192323e-02,  3.92902817e-04,
         1.22798001e-02, -2.46473104e-02,  6.86390772e-02,
        -3.56159210e-02,  6.47558719e-02,  4.92868163e-02,
        -6.31355420e-02, -5.60103282e-02, -4.72414568e-02,
        -9.81409755e-03,  3.59761193e-02,  7.56186917e-02,
         4.10142541e-02,  6.51928708e-02,  1.23051487e-01,
         1.34600443e-03, -4.19883057e-02,  1.54162794e-02,
        -1.00162067e-01, -5.18786646e-02, -8.33146423e-02,
         6.79359287e-02,  4.22359593e-02, -8.93923827e-03,
        -4.12749723e-02, -2.67390534e-02, -7.68720582e-02,
         1.46500096e-01,  3.16821150e-02, -3.74349691e-0

In [47]:
print(df.shape)
df.head()

(19997, 5)


,text,category,clean_text,embedding,cluster
0,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49...,alt.atheism,Archive name atheism resources Alt atheism arc...,"[-0.0285569429397583, 0.016917048022150993, -0...",4
1,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism,Archive name atheism introduction Alt atheism ...,"[-0.027238793671131134, -0.04237806797027588, ...",4
2,Newsgroups: alt.atheism\nPath: cantaloupe.srv....,alt.atheism,In article Charley Wingate writes Well John ha...,"[-0.09503985196352005, -0.004863628186285496, ...",4
3,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism,until kings become philosophers or philosopher...,"[-0.033383529633283615, 0.044307440519332886, ...",4
4,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...,alt.atheism,In article Bob McGwier writes 1 HOWEVER I hate...,"[-0.02208806946873665, 0.06549318879842758, -0...",4


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
def check_cache(query_embedding, threshold=0.90):

    for cached_query, cached_result in semantic_cache.items():

        similarity = cosine_similarity(
            [query_embedding],
            [cached_query]
        )[0][0]

        if similarity > threshold:
            return cached_result

    return None

In [52]:
import numpy as np

def semantic_search(query, k=5):

    print("Query:", query)

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for i in indices[0]:
        results.append({
            "category": df.iloc[i]["category"],
            "text": df.iloc[i]["clean_text"][:200]
        })

    return results

In [53]:
results = semantic_search("space shuttle launch mission")

for r in results:
    print(r["category"])
    print(r["text"])
    print("------")

Query: space shuttle launch mission
sci.space
Well you better not get the shuttle as your launch vehicle and most ELV s have too far of a backlog for political messages If during the campaign season the candidates for president had launched one r
------
sci.space
Archive name space schedule Last modified Date 93 04 01 14 39 23 SPACE SHUTTLE ANSWERS LAUNCH SCHEDULES TV COVERAGE SHUTTLE LAUNCHINGS AND LANDINGS SCHEDULES AND HOW TO SEE THEM Shuttle operations are
------
sci.space
For an essay I am writing about the space shuttle and a need for a better propulsion system Through research I have found that it is rather clumsy i e all the checks tests before launch the safety haz
------
sci.space
Sorry for asking a question that s not entirely based on the technical aspects of space but I couldn t find the answer on the FAQs I m currently in the UK which makes seeing a Space Shuttle launch a l
------
sci.space
This notice will be posted weekly in sci space sci astro and sci space shuttle The

In [54]:
!pip install fastapi uvicorn

In [55]:
df.to_csv("data/news_dataset.csv", index=False)

In [56]:
import numpy as np
np.save("data/embeddings.npy", embedding_matrix)

In [58]:
!pip install fastapi uvicorn
uvicorn app:app --reload

SyntaxError: invalid syntax (1876901537.py, line 2)